# 🏒 The Yost Feed: Regex, Text & Time

### SIADS 505 · Other Assignments · Assignment 3
**100 points — 10 questions × 10 points each**

---

Nobody hands you a tidy DataFrame. What they hand you is a **feed** — one row
per event, and the entire event crammed into a single string a scraper pulled
off a scoreboard:

```
GOAL 12:34 MICH #17 T. Brennan (A: #9 R. Karr, #22 D. Ochoa) [EV]
PENALTY 05:11 MICH #4 J. Ferris (Tripping, 2 min)
SHOT 8:45 OSU #22 P. Kuznetsov
```

Everything you want is in there. None of it is in a column.

This assignment is about getting it out: **regular expressions with named
groups**, `.str.extract`, `.str.findall`, `.explode`, clock arithmetic on a
clock that counts *down*, and the joins and group-wise counting that turn 4,065
lines of play-by-play into a scoring leaderboard.

> **Everything here is simulated.** One season, 41 games, 4,065 events, all
> generated from a fixed seed. Yost Ice Arena and the opponents are real;
> nothing else is. Do not cite any of it.

**How this is graded.** Ten questions, ten points each. The `assert`s printed
under each question are **worth zero points** — they exist so you can tell
whether you are on track. A separate set of hidden checks carries the grade,
and most of them re-run your function on a *different six games*. Write
functions that compute the answer; do not paste in the numbers you see below.

**A note on regex.** Every pattern you need here is fair. The traps are the
ones that show up in real feeds: whitespace the scraper left behind, surnames
containing an apostrophe or a hyphen, and infractions with spaces in them. A
pattern that works on the first ten rows and fails on row 900 is the whole
lesson.


## The data

Three CSV files live in `data/`.

### `data/events.csv` — one row per event, 4,065 rows

| column | what it is |
|---|---|
| `event_id` | `E00001`, unique, in chronological order |
| `game_id` | `2024-G07`, joins to `games.csv` |
| `period` | `1`, `2`, `3`, or **`4` for overtime** |
| `description` | the whole event, as one string |

**Every description has the same first five fields**, separated by whitespace:

```
TYPE  CLOCK  TEAM  #JERSEY  PLAYER
```

`TYPE` is one of `GOAL`, `SHOT`, `HIT`, `BLOCK`, `TAKEAWAY`, `PENALTY`.
`CLOCK` is **time remaining in the period**, `MM:SS` — and the leading zero is
missing about a third of the time (`8:45`, not `08:45`). `TEAM` is `MICH` or
the opponent's code. `PLAYER` is an initial and a surname.

Two event types carry more after that:

```
GOAL     ... (A: #9 R. Karr, #22 D. Ochoa) [EV]     assists, then the strength
GOAL     ... (unassisted) [PP]                      nobody gets an assist
PENALTY  ... (Tripping, 2 min)                      infraction, then the length
```

The strength tag is one of **`EV`** (even), **`PP`** (power play), **`SH`**
(short-handed), **`EN`** (empty net), **`PS`** (penalty shot).

**The feed was scraped, and it shows.** Some descriptions carry leading or
trailing spaces; some have a doubled space in the middle. That is not a typo in
this notebook — it is the assignment. Strip, and match runs of whitespace with
`\s+` rather than a single space.

### `data/games.csv` — one row per game, 41 rows

`game_id`, `season`, `game_no`, `date`, `opponent`, `opponent_code`, `site`,
`venue`, `michigan_goals`, `opponent_goals`, `result` (`W`/`L`), `overtime`,
`attendance`.

Simulated record: **23–18**, including **5 games in which Michigan did not
score at all**.

### `data/roster.csv` — one row per player, 26 rows

`jersey`, `player` (`"T. Brennan"`), `position` (`F`/`D`/`G`), `class`,
`shoots`, `height` (`"6-1"`), `weight_lb`, `hometown` (`"Toronto, ON"` or
`"Stockholm, Sweden"`).

Only Michigan players are on it. Opponent jersey numbers appear in the feed and
have no roster row — several of them collide with Michigan numbers, so filter
by `team` before you count anything.


---

## Setup

Run this first.


In [ ]:
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print("Puck drop.")


### Shared pieces

Every question below leans on the same three things: the pattern, the period
lengths, and one small helper that strips the scraped whitespace before
anything is matched against it.


In [ ]:
PERIOD_SECONDS = 20 * 60
OT_SECONDS = 5 * 60

STRENGTHS = ["EV", "PP", "SH", "EN", "PS"]

EVENT_PATTERN = (
    r"^(?P<event_type>[A-Z]+)\s+"
    r"(?P<clock>\d{1,2}:\d{2})\s+"
    r"(?P<team>[A-Z]+)\s+"
    r"#(?P<jersey>\d+)\s+"
    r"(?P<player>[^(]+)"
)


def _clean(events):
    """The feed is scraped; strip before anchoring a pattern to the start."""
    return events["description"].astype("object").str.strip()


### First contact

Read the feed before you write a pattern against it. This cell is ungraded.
Look in particular at the rows the scraper padded — `repr()` shows you the
whitespace that `print()` hides.


In [ ]:
raw = pd.read_csv("data/events.csv")

print(raw.shape)
display(raw.head(8))

print("\nA goal, a penalty, and an unassisted goal:")
for pattern in [r"GOAL.*\(A:", r"PENALTY", r"unassisted"]:
    hit = raw.loc[raw["description"].str.contains(pattern, regex=True), "description"].iloc[0]
    print(" ", repr(hit))

padded = raw["description"].str.strip() != raw["description"]
print(f"\nrows with stray whitespace: {padded.sum()}")
print(" ", repr(raw.loc[padded, "description"].iloc[0]))


---

## Question 1 — Warm-ups  ·  10 points

Three loaders. Nothing clever, but the shapes have to be right or every
question after this one is working from the wrong frame.

- **`load_events(path="data/events.csv")`** — index by **`event_id`**.
  **4,065 rows, 3 columns.** `period` should come back as an integer.
- **`load_games(path="data/games.csv")`** — parse **`date`** as a datetime,
  index by **`game_id`**. **41 rows, 12 columns.**
- **`load_roster(path="data/roster.csv")`** — read it as-is. **26 rows, 8
  columns**, with `jersey` still a **column**, not the index. (Question 6
  turns it into the index; the loader should not.)


In [ ]:
def load_events(path="data/events.csv") -> pd.DataFrame:
    events = pd.read_csv(path)
    return events.set_index("event_id")


def load_games(path="data/games.csv") -> pd.DataFrame:
    games = pd.read_csv(path, parse_dates=["date"])
    return games.set_index("game_id")


def load_roster(path="data/roster.csv") -> pd.DataFrame:
    return pd.read_csv(path)


**Public checks — worth 0 points. Run them.**


In [ ]:
events = load_events()
games = load_games()
roster = load_roster()

assert events.shape == (4065, 3), f"expected (4065, 3), got {events.shape}"
assert events.index.name == "event_id" and events.index.is_unique
assert list(events.columns) == ["game_id", "period", "description"]
assert sorted(events["period"].unique()) == [1, 2, 3, 4], "period 4 is overtime"

assert games.shape == (41, 12), f"expected (41, 12), got {games.shape}"
assert games.index.name == "game_id"
assert pd.api.types.is_datetime64_any_dtype(games["date"])
assert games["michigan_goals"].sum() == 143
assert (games["michigan_goals"] == 0).sum() == 5, "five simulated shutouts"

assert roster.shape == (26, 8), f"expected (26, 8), got {roster.shape}"
assert "jersey" in roster.columns, "leave jersey as a column here"

record = games["result"].value_counts()
print(f"🥅 Simulated season: {record['W']}-{record['L']}, "
      f"{games['michigan_goals'].sum()} goals for, {games['opponent_goals'].sum()} against.")
print("✅ Q1 public checks passed.")
events.head()


---

## Question 2 — Reading the sheet  ·  10 points

One pattern, five columns.

Write **`parse_events(events)`** returning a DataFrame on the **same index**
with exactly these columns, in this order:

| column | example |
|---|---|
| `event_type` | `GOAL` |
| `clock` | `12:34` — the string, exactly as the feed wrote it |
| `team` | `MICH` |
| `jersey` | `17` — an **integer**, no `#` |
| `player` | `T. Brennan` — **stripped**, and *not* including the assists |

**All 4,065 rows must parse.** Two things will bite you:

1. Some descriptions have **leading whitespace**. A pattern anchored with `^`
   will not match them until you `.str.strip()` first.
2. Some have a **doubled space** in the middle. Match `\s+`, not `" "`.

*Hint:* `.str.extract()` with **named groups** — `(?P<event_type>...)` — gives
you a DataFrame with the group names as columns in one call. Useful pieces:
`[A-Z]+` for the type and team, `\d{1,2}:\d{2}` for the clock (the leading
zero is not always there), `#(?P<jersey>\d+)` for the number, and for the
player, "everything up to the first `(`, or the end of the string" —
`(?P<player>[^(]+)` — then `.str.strip()` the result.


In [ ]:
def parse_events(events: pd.DataFrame) -> pd.DataFrame:
    out = _clean(events).str.extract(EVENT_PATTERN)
    out["jersey"] = pd.to_numeric(out["jersey"]).astype("int64")
    out["player"] = out["player"].str.strip()
    return out[["event_type", "clock", "team", "jersey", "player"]]


**Public checks — worth 0 points. Run them.**


In [ ]:
parsed = parse_events(events)

assert list(parsed.columns) == ["event_type", "clock", "team", "jersey", "player"]
assert len(parsed) == 4065 and parsed.index.equals(events.index)
assert parsed["event_type"].isna().sum() == 0, "some rows did not parse — check the whitespace"
assert parsed["jersey"].dtype.kind in "iu", f"jersey should be an int, got {parsed['jersey'].dtype}"

assert parsed["event_type"].value_counts().to_dict() == {
    "SHOT": 2114, "HIT": 687, "BLOCK": 480, "PENALTY": 360, "GOAL": 256, "TAKEAWAY": 168,
}
assert (parsed["team"] == "MICH").sum() == 1995
assert parsed.loc["E00001", "player"] == "A. Quintero"
assert not parsed["player"].str.contains(r"[()#]").any(), "the player field picked up extra text"
assert (parsed["player"] == parsed["player"].str.strip()).all(), "strip the player names"

print("📋 Event types in the feed:")
print(parsed["event_type"].value_counts().to_string())
print("✅ Q2 public checks passed.")
parsed.head()


---

## Question 3 — Clock math  ·  10 points

A hockey clock **counts down**. `19:16` in the first period is 44 seconds into
the game, not nineteen minutes in. Every "when did this happen" question in
this notebook needs one honest number instead.

Write **`elapsed_seconds(events)`** returning an **integer** Series named
**`elapsed_seconds`**: seconds since the opening face-off.

- **Regulation periods are 20 minutes (1,200 seconds).**
- **Overtime — period 4 — is 5 minutes (300 seconds).**

So for a regulation period `p`:

```
(p - 1) × 1200  +  (1200 − seconds_remaining)
```

and for period 4, the game already has 3,600 seconds of regulation behind it
and the period itself is only 300 seconds long. Getting overtime wrong is worth
3 of the 10 points on its own.

*Hint:* you already have the clock string from Question 2 — split it on `":"`
and convert both halves. `np.where(period == 4, 300, 1200)` gives you the
period length as an array you can do arithmetic with.


In [ ]:
def elapsed_seconds(events: pd.DataFrame) -> pd.Series:
    clock = parse_events(events)["clock"]
    parts = clock.str.split(":", expand=True)
    remaining = pd.to_numeric(parts[0]) * 60 + pd.to_numeric(parts[1])

    period = events["period"].astype("int64")
    length = np.where(period == 4, OT_SECONDS, PERIOD_SECONDS)
    out = ((period - 1) * PERIOD_SECONDS + (length - remaining)).astype("int64")
    out.name = "elapsed_seconds"
    return out


**Public checks — worth 0 points. Run them.**


In [ ]:
elapsed = elapsed_seconds(events)

assert isinstance(elapsed, pd.Series) and elapsed.name == "elapsed_seconds"
assert elapsed.dtype.kind in "iu", f"whole seconds, got {elapsed.dtype}"
assert len(elapsed) == 4065 and elapsed.index.equals(events.index)
assert elapsed.min() > 0, "the clock counts down"

assert elapsed.loc["E00001"] == 43, "19:17 left in the first period is 43 seconds in"
assert elapsed.sum() == 7453622

# Overtime is five minutes long, and starts at 3,600.
overtime = events["period"] == 4
assert overtime.sum() == 43
assert elapsed[overtime].min() >= 3600 and elapsed[overtime].max() <= 3900, \
    "an overtime period is 300 seconds, not 1200"

print(f"⏱️  Latest event in the season: {elapsed.max()} seconds "
      f"({elapsed.max() // 60}:{elapsed.max() % 60:02d} of hockey).")
print("✅ Q3 public checks passed.")
elapsed.head()


---

## Question 4 — Assist credits  ·  10 points

A goal line can name nobody, one player, or two:

```
GOAL 12:34 MICH #17 T. Brennan (A: #9 R. Karr, #22 D. Ochoa) [EV]
GOAL 03:02 MICH #10 E. Marchand (unassisted) [EV]
```

Write **`assist_credits(events)`** returning an **integer** Series named
**`assists`**, indexed by **`jersey`** (index named `jersey`), ascending, with
one row per jersey that earned at least one assist.

**Michigan goals only.** Opponent jersey numbers overlap Michigan's, so a
count that ignores `team` is wrong in a way that still looks plausible — one of
the hidden checks looks specifically for jerseys that are not on the roster.

*Hint:* this is the one question where `.str.findall()` beats `.str.extract()`.
`extract` gives you the *first* match; `findall(r"#(\d+)")` gives you **all**
of them, as a list per row. Then `.explode()` turns a Series of lists into one
row per element, and `.value_counts()` finishes the job. Pull the parenthesised
credit out first so a scorer's own number is not counted as an assist.


In [ ]:
def assist_credits(events: pd.DataFrame) -> pd.Series:
    parsed = parse_events(events)
    michigan_goals = (parsed["event_type"] == "GOAL") & (parsed["team"] == "MICH")

    credits = _clean(events)[michigan_goals].str.extract(r"\(A:([^)]*)\)")[0]
    jerseys = credits.dropna().str.findall(r"#(\d+)").explode()

    out = jerseys.astype("int64").value_counts().sort_index().astype("int64")
    out.name = "assists"
    out.index.name = "jersey"
    return out


**Public checks — worth 0 points. Run them.**


In [ ]:
assists = assist_credits(events)

assert isinstance(assists, pd.Series) and assists.name == "assists"
assert assists.index.name == "jersey"
assert assists.dtype.kind in "iu", "assists are whole numbers"
assert list(assists.index) == sorted(assists.index), "sort by jersey number"

on_roster = set(roster["jersey"])
assert set(assists.index) <= on_roster, "some of those jerseys are not Michigan players"

assert assists.sum() == 203, "203 assists were handed out on Michigan goals"
assert len(assists) == 24
assert assists.loc[9] == 16 and assists.idxmax() == 9

top = assists.sort_values(ascending=False).head(3)
names = roster.set_index("jersey")["player"]
print("🍎 Most assists:")
for jersey, count in top.items():
    print(f"   {count:>2}  #{jersey} {names[jersey]}")
print("✅ Q4 public checks passed.")
assists.head()


---

## Question 5 — The penalty box  ·  10 points

```
PENALTY 05:11 MICH #4 J. Ferris (Tripping, 2 min)
PENALTY 12:40 MICH #22 F. Delacroix (Delay of Game, 2 min)
```

Two fields inside one set of parentheses, and the first one has spaces and
hyphens in it (`Delay of Game`, `Cross-checking`, `Contact to the Head`). A
pattern that grabs one word gets three of the thirteen infractions right.

Write **`penalty_summary(events)`** — **Michigan penalties only** — returning a
DataFrame indexed by **`infraction`** with two **integer** columns:

| column | definition |
|---|---|
| `count` | how many times it was called |
| `minutes` | total penalty minutes for it |

Sorted by **`minutes` descending**, ties broken by **`infraction`
alphabetically**.

*Hint:* one `.str.extract()` with two named groups does both fields at once —
`\((?P<infraction>[^,]+),\s*(?P<minutes>\d+)\s*min\)`. "Everything up to the
comma" is the trick that keeps the multi-word names intact. Then
`groupby("infraction").agg(...)` and a two-key `sort_values`.


In [ ]:
def penalty_summary(events: pd.DataFrame) -> pd.DataFrame:
    parsed = parse_events(events)
    taken = (parsed["event_type"] == "PENALTY") & (parsed["team"] == "MICH")

    detail = _clean(events)[taken].str.extract(
        r"\((?P<infraction>[^,]+),\s*(?P<minutes>\d+)\s*min\)"
    )
    detail["minutes"] = pd.to_numeric(detail["minutes"])

    out = detail.groupby("infraction").agg(
        count=("minutes", "size"), minutes=("minutes", "sum")
    )
    out = out.astype("int64").reset_index()
    out = out.sort_values(["minutes", "infraction"], ascending=[False, True])
    return out.set_index("infraction")


**Public checks — worth 0 points. Run them.**


In [ ]:
box = penalty_summary(events)

assert list(box.columns) == ["count", "minutes"]
assert box.index.name == "infraction"
assert box["count"].dtype.kind in "iu" and box["minutes"].dtype.kind in "iu"

for label in ["Delay of Game", "Too Many Men", "Cross-checking", "Contact to the Head"]:
    assert label in box.index, f"missing multi-word infraction: {label}"

assert len(box) == 13
assert box["minutes"].sum() == 357
assert box.index[0] == "Tripping"
assert box.loc["Tripping", "count"] == 33 and box.loc["Tripping", "minutes"] == 66
assert box["minutes"].is_monotonic_decreasing, "sort by minutes, descending"

print(f"🚨 {box['count'].sum()} Michigan penalties, {box['minutes'].sum()} minutes.")
print(box.head(5).to_string())
print("✅ Q5 public checks passed.")
box


---

## Question 6 — Roster details  ·  10 points

Three small string problems, all of which have an obvious wrong answer.

Write **`roster_details(roster)`** returning a DataFrame indexed by
**`jersey`** (ascending, index named `jersey`) with these columns:

| column | from | note |
|---|---|---|
| `last_name` | `player` | `"T. Brennan"` → `"Brennan"` |
| `height_in` | `height` | `"6-1"` → `73`, an **integer** |
| `home_region` | `hometown` | `"Toronto, ON"` → `"ON"` |

The traps, all three of them on this roster:

- **`"R. St. Pierre"` → `"St. Pierre"`.** Taking "the second word" gives you
  `"St."`. Take *everything after the first initial*.
- `"L. O'Halloran"` and `"P. Vander-Meer"` are the same trap wearing an
  apostrophe and a hyphen.
- **`"Stockholm, Sweden"` → `"Sweden"`.** Not every player is from a place with
  a two-letter code, so "the last two characters" is wrong. Split on the last
  comma.

*Hint:* `str.extract(r"^\S+\s+(.+)$")` for the surname,
`.str.split("-", expand=True)` for the height, and `.str.rsplit(",", n=1)` —
note the **`r`** — for the hometown.


In [ ]:
def roster_details(roster: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=pd.Index(roster["jersey"].astype("int64"), name="jersey"))

    out["last_name"] = roster["player"].str.extract(r"^\S+\s+(.+)$")[0].to_numpy()

    height = roster["height"].str.split("-", expand=True)
    feet = pd.to_numeric(height[0])
    inches = pd.to_numeric(height[1])
    out["height_in"] = (feet * 12 + inches).astype("int64").to_numpy()

    out["home_region"] = (
        roster["hometown"].str.rsplit(",", n=1).str[1].str.strip().to_numpy()
    )
    return out.sort_index()


**Public checks — worth 0 points. Run them.**


In [ ]:
details = roster_details(roster)

assert list(details.columns) == ["last_name", "height_in", "home_region"]
assert details.index.name == "jersey" and len(details) == 26
assert details["height_in"].dtype.kind in "iu", "whole inches"

assert details.loc[8, "last_name"] == "St. Pierre", "a surname can contain a period"
assert details.loc[6, "last_name"] == "O'Halloran"
assert details.loc[2, "last_name"] == "Vander-Meer"

assert details.loc[2, "height_in"] == 76, "6-4 is 76 inches"
assert details.loc[27, "height_in"] == 70

assert details.loc[4, "home_region"] == "ON"
assert details.loc[14, "home_region"] == "Sweden", "not every hometown ends in a state code"
assert details.loc[29, "home_region"] == "Philippines"

print("🌍 Where this simulated roster comes from:")
print(details["home_region"].value_counts().head(6).to_string())
print("✅ Q6 public checks passed.")
details.head()


---

## Question 7 — Scoring leaders  ·  10 points

Two counts, one merge, one sort — the shape of most reporting jobs.

Write **`scoring_leaders(events, roster)`** returning a DataFrame indexed by
**`player`** (the full name from the roster, index named `player`) with
columns, in this order:

| column | definition |
|---|---|
| `jersey` | the player's number |
| `goals` | Michigan goals they scored |
| `assists` | assists they were credited with |
| `points` | `goals + assists` |

Include every Michigan player with **at least one point**. A player who scored
but never assisted has `0` assists, **not `NaN`**, and vice versa.

Sorted by **`points` descending**, then **`goals` descending**, then **`player`
alphabetically**. All three counts are **integers**.

*Hint:* count goals from the parsed frame, reuse `assist_credits` for the
assists, put the two Series side by side in one `pd.DataFrame({...})` — pandas
aligns them on the jersey index for you and leaves `NaN` where one side has no
row, which is what `.fillna(0)` is for. Then merge the roster on `jersey` to
get the names.


In [ ]:
def scoring_leaders(events: pd.DataFrame, roster: pd.DataFrame) -> pd.DataFrame:
    parsed = parse_events(events)
    michigan_goals = (parsed["event_type"] == "GOAL") & (parsed["team"] == "MICH")

    goals = parsed.loc[michigan_goals, "jersey"].value_counts()
    assists = assist_credits(events)

    tally = pd.DataFrame({"goals": goals, "assists": assists}).fillna(0).astype("int64")
    tally.index.name = "jersey"
    tally["points"] = tally["goals"] + tally["assists"]
    tally = tally[tally["points"] > 0].reset_index()

    merged = tally.merge(roster[["jersey", "player"]], on="jersey", how="left")
    merged = merged.sort_values(
        ["points", "goals", "player"], ascending=[False, False, True]
    )
    return merged.set_index("player")[["jersey", "goals", "assists", "points"]]


**Public checks — worth 0 points. Run them.**


In [ ]:
leaders = scoring_leaders(events, roster)

assert list(leaders.columns) == ["jersey", "goals", "assists", "points"]
assert leaders.index.name == "player"
assert (leaders["points"] == leaders["goals"] + leaders["assists"]).all()
assert leaders[["goals", "assists", "points"]].dtypes.map(lambda d: d.kind).isin(["i", "u"]).all()

assert leaders["goals"].sum() == 143, "every Michigan goal belongs to somebody"
assert leaders["assists"].sum() == 203
assert len(leaders) == 24
assert leaders.index[0] == "R. Karr" and leaders.iloc[0]["points"] == 26

# Tied on points, more goals is listed first.
tied = leaders[leaders["points"] == 20]
assert list(tied.index) == ["T. Brennan", "K. Nystrom"], "ties break on goals"

print("🏒 Simulated scoring leaders:")
print(leaders.head(5).to_string())
print("✅ Q7 public checks passed.")
leaders.head(10)


---

## Question 8 — Special teams  ·  10 points

Every goal line ends with a strength tag in square brackets: **`EV`** even
strength, **`PP`** power play, **`SH`** short-handed, **`EN`** empty net,
**`PS`** penalty shot.

Write **`strength_breakdown(events)`** returning a DataFrame of **integer**
counts:

- rows: exactly **`["Michigan", "Opponent"]`** — every non-`MICH` team code
  folds into `Opponent`,
- columns: exactly **`["EV", "PP", "SH", "EN", "PS"]`**, in that order,
- index named `team`, columns named `strength`.

**Nobody scored on a penalty shot this season.** `crosstab` and `pivot_table`
will not invent a column for a value that never appears, so `PS` will be
missing unless you put it back — as a column of **zeroes**. A count of zero is
a fact; a missing column is a bug waiting to crash whatever reads this table
next.

*Hint:* extract the tag with `\[(\w+)\]`, build the two-value team label with
`np.where(...)`, `pd.crosstab` the two, then
`.reindex(index=[...], columns=[...], fill_value=0)`.


In [ ]:
def strength_breakdown(events: pd.DataFrame) -> pd.DataFrame:
    parsed = parse_events(events)
    goals = parsed["event_type"] == "GOAL"

    tag = _clean(events)[goals].str.extract(r"\[(\w+)\]")[0]
    tag.name = "strength"
    side = pd.Series(
        np.where(parsed.loc[goals, "team"] == "MICH", "Michigan", "Opponent"),
        index=tag.index,
        name="team",
    )

    table = pd.crosstab(side, tag)
    table = table.reindex(
        index=["Michigan", "Opponent"], columns=STRENGTHS, fill_value=0
    ).astype("int64")
    table.index.name = "team"
    table.columns.name = "strength"
    return table


**Public checks — worth 0 points. Run them.**


In [ ]:
special = strength_breakdown(events)

assert list(special.index) == ["Michigan", "Opponent"]
assert list(special.columns) == ["EV", "PP", "SH", "EN", "PS"]
assert special.index.name == "team" and special.columns.name == "strength"
assert special.to_numpy().dtype.kind in "iu", "counts, not floats"

assert special.loc["Michigan"].sum() == 143
assert special.loc["Opponent"].sum() == 113
assert special.loc["Michigan", "PP"] == 33
assert (special["PS"] == 0).all(), "no penalty-shot goals — but the column stays"

print("⚡ Goals by manpower situation:")
print(special.to_string())
pp_share = special.loc["Michigan", "PP"] / special.loc["Michigan"].sum()
print(f"\n{pp_share:.1%} of Michigan's simulated goals came on the power play.")
print("✅ Q8 public checks passed.")
special


---

## Question 9 — First blood  ·  10 points

Write **`first_goal_times(events, games)`** returning a **float** Series named
**`first_goal_second`**:

- indexed by **`game_id`**, **exactly matching `games` — same rows, same
  order**,
- the value is `elapsed_seconds` of **Michigan's first goal** in that game,
- and **`NaN` for the five games Michigan never scored**.

That last point is the question. `groupby(...).min()` only produces rows for
games that *had* a Michigan goal, so five of them simply will not be there. A
`reindex` against `games.index` puts them back as `NaN`, which is the honest
answer: there was no first goal, and `0` would claim they scored on the opening
face-off.

*Hint:* filter to Michigan goals, group the elapsed seconds by `game_id`, take
`.min()`, then `.reindex(games.index)`. The float dtype comes for free once
there are `NaN`s in it.


In [ ]:
def first_goal_times(events: pd.DataFrame, games: pd.DataFrame) -> pd.Series:
    parsed = parse_events(events)
    michigan_goals = (parsed["event_type"] == "GOAL") & (parsed["team"] == "MICH")

    elapsed = elapsed_seconds(events)[michigan_goals]
    first = elapsed.groupby(events.loc[michigan_goals, "game_id"]).min()

    out = first.reindex(games.index).astype(float)
    out.name = "first_goal_second"
    out.index.name = "game_id"
    return out


**Public checks — worth 0 points. Run them.**


In [ ]:
first = first_goal_times(events, games)

assert isinstance(first, pd.Series) and first.name == "first_goal_second"
assert first.index.equals(games.index), "same rows, same order as games"
assert pd.api.types.is_float_dtype(first), "a shutout has no first goal"

shutouts = games.index[games["michigan_goals"] == 0]
assert len(shutouts) == 5
assert first.loc[shutouts].isna().all(), "shut out means NaN, not 0"
assert first.isna().sum() == 5

assert first.min() == 78.0 and first.idxmin() == "2024-G07"
assert abs(first.mean() - 1024.78) < 0.01, ".mean() skips the NaNs for you"

quick = first.idxmin()
print(f"⚡ Fastest simulated opening goal: {int(first.min())} seconds, in {quick} "
      f"vs {games.loc[quick, 'opponent']}.")
print(f"   Median wait for the first goal: {first.median():.0f} seconds.")
print("✅ Q9 public checks passed.")
first.head()


---

## Question 10 — The game winner  ·  10 points

Last one, and it is the only question here that needs a real idea rather than a
real pattern.

Hockey defines the **game-winning goal** precisely: in a game the winners took
`N` goals to `M`, the game-winner is the winning team's **(M + 1)-th goal**.
Win 4–2 and it is your third goal; win 1–0 and it is your first. It is the goal
after which the loser could no longer catch up.

Write **`game_winning_goals(events, games)`** returning a DataFrame indexed by
**`game_id`** (ascending) with **one row per Michigan win** — 23 of them — and
columns, in this order:

| column | |
|---|---|
| `period` | integer, `4` if it came in overtime |
| `clock` | the string from the feed |
| `jersey` | integer |
| `player` | as the feed wrote it, e.g. `T. Brennan` |

Losses do not appear at all.

*Hint:* take Michigan's goals, sort them by `elapsed_seconds` — **not by
`event_id`, and not by clock**, which counts down and resets three times a
night — and number them within each game with
`groupby("game_id").cumcount() + 1`. Then join that against
`opponent_goals + 1` for the games Michigan won and keep the rows where the two
agree.


In [ ]:
def game_winning_goals(events: pd.DataFrame, games: pd.DataFrame) -> pd.DataFrame:
    parsed = parse_events(events)
    michigan_goals = (parsed["event_type"] == "GOAL") & (parsed["team"] == "MICH")

    scored = parsed.loc[michigan_goals, ["clock", "jersey", "player"]].copy()
    scored["game_id"] = events.loc[michigan_goals, "game_id"]
    scored["period"] = events.loc[michigan_goals, "period"].astype("int64")
    scored["_elapsed"] = elapsed_seconds(events)[michigan_goals]

    scored = scored.sort_values(["game_id", "_elapsed"])
    scored["_goal_no"] = scored.groupby("game_id").cumcount() + 1

    wins = games[games["michigan_goals"] > games["opponent_goals"]]
    target = (wins["opponent_goals"] + 1).rename("_target").to_frame()

    merged = scored.merge(target, left_on="game_id", right_index=True, how="inner")
    winners = merged[merged["_goal_no"] == merged["_target"]]

    out = winners.set_index("game_id")[["period", "clock", "jersey", "player"]]
    return out.sort_index()


**Public checks — worth 0 points. Run them.**


In [ ]:
winners = game_winning_goals(events, games)

assert list(winners.columns) == ["period", "clock", "jersey", "player"]
assert winners.index.name == "game_id"

wins = games.index[games["result"] == "W"]
assert len(winners) == 23 and set(winners.index) == set(wins), "one row per win, no losses"
assert list(winners.index) == sorted(winners.index)
assert winners["period"].dtype.kind in "iu" and winners["jersey"].dtype.kind in "iu"

assert winners.loc["2024-G01", "player"] == "V. Sepp"
assert winners.loc["2024-G01", "clock"] == "09:03"
assert winners.loc["2024-G13", "period"] == 4, "that one went to overtime"
assert (winners["period"] == 4).sum() == 3, "three simulated overtime winners"

print("🏆 Most game-winning goals:")
print(winners["player"].value_counts().head(3).to_string())
print("\nThe three overtime winners:")
print(winners[winners["period"] == 4].to_string())
print("✅ Q10 public checks passed — that's all ten. Go Blue. 〽️")
winners.head()


---

# 🎉 Victory lap

That's 100 points. Everything below is **ungraded** — no asserts, no hidden
tests, nothing to hand in. Once the feed is parsed, it will answer almost
anything you ask it.


In [ ]:
# Everything in one frame: the parsed feed, the clock, and the game context.
feed = (
    parse_events(events)
    .assign(elapsed=elapsed_seconds(events),
            period=events["period"],
            game_id=events["game_id"])
    .join(games[["opponent", "site", "result"]], on="game_id")
)
feed["side"] = np.where(feed["team"] == "MICH", "Michigan", "Opponent")
feed.head()


In [ ]:
# When do goals happen? Five-minute buckets across regulation.
regulation = feed.query("event_type == 'GOAL' and period <= 3").copy()
regulation["bucket"] = pd.cut(
    regulation["elapsed"], bins=range(0, 3601, 300),
    labels=[f"{i}-{i + 5}" for i in range(0, 60, 5)],
)
pd.crosstab(regulation["bucket"], regulation["side"])


In [ ]:
# Shot volume versus goals — the crudest possible shooting percentage.
shots = feed.query("event_type in ['SHOT', 'GOAL']")
shots.groupby("side").agg(
    attempts=("event_type", "size"),
    goals=("event_type", lambda s: (s == "GOAL").sum()),
).assign(pct=lambda d: (d["goals"] / d["attempts"]).round(3))


In [ ]:
# Does taking penalties cost you? Michigan PIM per game against the result.
pim = (
    feed.query("event_type == 'PENALTY' and side == 'Michigan'")
    .join(events["description"].str.extract(r"(?P<mins>\d+)\s*min\)"), how="left")
    .assign(mins=lambda d: pd.to_numeric(d["mins"]))
    .groupby("game_id")["mins"].sum()
)
summary = games.assign(pim=pim).fillna({"pim": 0})
summary.groupby("result")["pim"].agg(["size", "mean"]).round(2)


In [ ]:
# A picture, if matplotlib is installed.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed — try: pip install matplotlib")
else:
    goals = feed.query("event_type == 'GOAL' and period <= 3")
    fig, ax = plt.subplots(figsize=(11, 4))
    bins = range(0, 3601, 120)
    ax.hist(goals.loc[goals["side"] == "Michigan", "elapsed"], bins=bins,
            color="#00274C", alpha=0.85, label="Michigan")
    ax.hist(goals.loc[goals["side"] == "Opponent", "elapsed"], bins=bins,
            color="#FFCB05", alpha=0.65, label="Opponent")
    for boundary in (1200, 2400):
        ax.axvline(boundary, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Simulated goals by time of game")
    ax.set_xlabel("seconds since the opening face-off")
    ax.set_ylabel("goals")
    ax.legend()
    plt.tight_layout()
    plt.show()


---

### Your turn

Add a cell and ask the feed something nobody assigned. It can answer all of
these:

- Who **shoots the most** relative to how much they score, and is that a
  criticism?
- Do goals cluster in the **last two minutes** of a period? Of the third?
- Which pairs of jerseys show up on the **same goal line** most often — the
  simulated top line?
- Michigan takes penalties in bunches. Is a penalty **more likely right after**
  the other team's? (`elapsed`, `shift`, and a threshold.)
- The strength tag says `EN` for empty-net goals. What does the **time
  distribution** of those look like, and does it make sense?
- Write the regex from Question 2 as `re.compile(..., re.VERBOSE)` with a
  comment on each group. Which version would you rather find in a codebase in
  six months?

---

<div align="center">

**〽️ Go Blue.**

*Hail to the victors, and to everyone who tested their pattern on row 900.*

</div>
